这是**整数规划 (Integer Programming, IP)** 与 **0-1规划** 的详细解析。

虽然它们属于线性规划的“亲兄弟”，但**0-1规划**是建模比赛中**最考验逻辑**的模型。它能把复杂的“逻辑关系”（如：选A就不能选B、至少选3个、必须同时开始）转化成数学公式。

---

### 一、 算法原理
**核心思想**：**“在格子上跳舞”与“是或否的抉择”。**

1.  **整数规划**：线性规划的变体，要求变量必须是**整数**（比如：你不能建 1.5 个工厂，也不能派 3.4 个人）。
2.  **0-1规划**：变量只能取 **0 或 1**。
    *   **1** 代表：选、做、买、True。
    *   **0** 代表：不选、不做、不买、False。
3.  **求解难度**：比线性规划难得多！因为不能用单纯形法直接求，通常使用**分支定界法 (Branch and Bound)** 或 **割平面法**。好在现在的求解器（Solver）已经足够强大，我们只需要负责建模。

---

### 二、 常见逻辑模型的数学表达 (必看!)
0-1规划的精髓在于**把逻辑变成公式**。假设 $x_i$ 是 0-1 变量：

1.  **互斥约束**（A和B只能选一个）：
    $$ x_A + x_B \le 1 $$
2.  **依赖约束**（选了A必须选B，但选B不一定选A）：
    $$ x_A \le x_B $$
3.  **至少选N个**：
    $$ \sum x_i \ge N $$
4.  **固定成本问题**（只要生产 $y>0$，就必须付一笔开工费 $K$）：
    引入0-1变量 $x$。约束：$y \le M \cdot x$ （$M$是一个很大的数）。
    *   如果 $x=0$，则 $y$ 必须为0（不付钱就不准生产）。
    *   如果 $x=1$，则 $y \le M$（只要不超过M都能生产）。

---

### 三、 适用场景
1.  **选址问题**：在10个备选点中选3个建仓库，总成本最低。
2.  **排班问题**：护士排班，满足每人工作时长限制，且每班次人数足够。
3.  **背包问题**：容量有限，怎么装价值最大？
4.  **指派问题**：N个人干N个活，怎么分配效率最高？

---

### 四、 Python 代码框架

对于整数/0-1规划，强烈推荐使用 **`PuLP`** 库，而不是 `scipy`。
**原因**：`scipy` 需要你写巨大的矩阵，处理逻辑约束（如“选A必须选B”）非常痛苦。而 `PuLP` 可以像写英语一样写约束，非常适合建模逻辑复杂的题目。

先安装：`pip install pulp`

```python
import pulp

def solve_integer_programming():
    """
    0-1 规划与整数规划求解模板 (使用 PuLP)

    【场景模拟】：项目投资选择
    有 5 个项目 (A, B, C, D, E)，每个项目有投入成本和预期收益。
    资金有限，且有逻辑约束。

    数据：
    项目:   A    B    C    D    E
    成本:   10   20   30   15   25  (万元)
    收益:   15   35   50   20   40  (万元)

    约束：
    1. 总资金不超过 60 万元。
    2. A 和 B 是互斥的 (技术冲突，只能二选一，或都不选)。
    3. 选了 C 就必须选 D (D是C的配套设施)。
    4. E 项目如果选，必须建立在 A 项目已选的基础上 (A是E的前置)。
    """

    # 1. 定义问题
    # pulp.LpMaximize (求最大值) 或 pulp.LpMinimize (求最小值)
    prob = pulp.LpProblem("Project_Selection", pulp.LpMaximize)

    # 2. 定义变量
    # cat='Binary' 表示 0-1 变量
    # cat='Integer' 表示 整数变量
    # cat='Continuous' 表示 连续变量
    projects = ['A', 'B', 'C', 'D', 'E']
    costs = {'A': 10, 'B': 20, 'C': 30, 'D': 15, 'E': 25}
    profits = {'A': 15, 'B': 35, 'C': 50, 'D': 20, 'E': 40}

    # 创建一个字典，包含所有决策变量 x_A, x_B ...
    x = pulp.LpVariable.dicts("Select", projects, cat='Binary')

    # 3. 定义目标函数 (直接加到 prob 中)
    # 目标：最大化总收益
    prob += pulp.lpSum([profits[i] * x[i] for i in projects]), "Total_Profit"

    # 4. 定义约束条件

    # (1) 资金限制
    prob += pulp.lpSum([costs[i] * x[i] for i in projects]) <= 60, "Budget_Limit"

    # (2) A 和 B 互斥 (A + B <= 1)
    prob += x['A'] + x['B'] <= 1, "Mutually_Exclusive_A_B"

    # (3) 选 C 必选 D (C <= D  =>  若C=1则D必为1)
    prob += x['C'] <= x['D'], "C_requires_D"

    # (4) 选 E 必须以 A 为前提 (E <= A => 若E=1则A必为1; 若A=0则E必为0)
    prob += x['E'] <= x['A'], "E_requires_A"

    # 5. 求解
    # 使用默认求解器 (CBC)
    # msg=1 显示求解过程, msg=0 不显示
    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    # 6. 输出结果
    print("-" * 30)
    print("求解状态:", pulp.LpStatus[prob.status])

    if pulp.LpStatus[prob.status] == 'Optimal':
        print("最大收益:", pulp.value(prob.objective))
        print("被选中的项目:")
        for i in projects:
            if pulp.value(x[i]) == 1:
                print(f"  - 项目 {i} (成本:{costs[i]}, 收益:{profits[i]})")

        # 验证一下资金
        total_cost = sum([costs[i] for i in projects if pulp.value(x[i]) == 1])
        print(f"实际总花费: {total_cost} 万元")
    else:
        print("无可行解")

# ================= 使用示例 =================

if __name__ == "__main__":
    solve_integer_programming()
```

### 代码使用的“修修补补”指南：

1.  **如何把 0-1 变量改成整数变量？**
    *   假设题目变了：每个项目不是“选或不选”，而是可以“投资 0 到 5 份”。
    *   **修改**：
        ```python
        # lowBound=0, upBound=5 (上下界)
        x = pulp.LpVariable.dicts("Invest_Num", projects, lowBound=0, upBound=5, cat='Integer')
        ```

2.  **处理“如果...那么...” (Big-M 法)**
    *   题目：如果生产数量 $x > 0$，则必须支付固定成本 1000 元。
    *   **技巧**：
        *   定义连续变量 $x$ (产量)。
        *   定义 0-1 变量 $y$ (是否开工)。
        *   添加约束：$x \le M \cdot y$ （$M$ 是一个很大的数，比如 100000）。
        *   目标函数里减去 $1000 \cdot y$。
        *   *解释*：如果 $y=0$ (不开工)，公式变成 $x \le 0$，强制 $x=0$。如果 $y=1$，公式变成 $x \le 100000$，$x$ 就可以自由生产了。

3.  **多目标规划怎么弄？**
    *   题目：既要收益最大，又要风险最小。
    *   **方法一（线性加权）**：
        `prob += 0.7 * 收益 - 0.3 * 风险`
    *   **方法二（分层序列法）**：
        1. 先算收益最大，算出最大收益是 100万。
        2. 再建立一个新模型，求风险最小。添加约束：`收益 >= 100万 * 0.9` (允许收益稍微少一点点)。

4.  **求解速度太慢？**
    *   整数规划是 NP-hard 问题，变量一多（比如几百个 0-1 变量），可能跑一天都跑不完。
    *   **设置时间限制**：
        `prob.solve(pulp.PULP_CBC_CMD(timeLimit=60))` (单位：秒)
    *   这样求解器会在 60秒 后停止，并返回当前找到的**最好**的解（虽然可能不是理论最优，但在比赛中足够了）。